# NER Model Training & Evaluation

## Domain-Adapted NER for Enron Emails

This notebook builds a domain-adapted Named Entity Recognition (NER) pipeline for Enron emails.

### Approach
1. Load the processed Enron email dataset.
2. Run baseline spaCy NER.
3. Detect Enron directory-style person names.
4. Mask those directory expressions before spaCy NER.
5. Combine Enron-specific PERSON entities with spaCy entities.
6. Remove duplicates.
7. Evaluate baseline vs. domain-adapted NER using the manually prepared ground truth.

> **Note:** The current notebook uses a pretrained spaCy NER model plus rule-based Enron-specific extraction. It does not fine-tune/train the spaCy model weights.

## 1. Setup

In [ ]:
!pip install -q spacy pandas
!python -m spacy download en_core_web_sm

In [ ]:
import os
import re
import pandas as pd
import spacy

In [ ]:
nlp = spacy.load("en_core_web_sm")
print("spaCy version:", spacy.__version__)
print("NER model loaded successfully!")

## 2. Load Dataset

In [ ]:
BASE_PATH = "/kaggle/input/datasets/prishabhkumar/processed-data/processed"

TRAIN_PATH = os.path.join(BASE_PATH, "train", "intent_train.csv")
VAL_PATH = os.path.join(BASE_PATH, "val", "intent_val.csv")
TEST_PATH = os.path.join(BASE_PATH, "test", "intent_test.csv")

In [7]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (420, 4)
Validation: (90, 4)
Test: (90, 4)


## 3. Define Baseline NER

In [ ]:
ENTITY_TYPES = [
    "PERSON",
    "ORG",
    "DATE",
    "GPE",
    "MONEY",
    "TIME",
    "EVENT"
]

def clean_entity_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def format_entities(entities):
    return "; ".join(
        f"{e['text']} ({e['label']})"
        for e in entities
    )

def extract_entities(text):
    if not isinstance(text, str):
        return []

    doc = nlp(text)

    return [
        {
            "text": clean_entity_text(ent.text),
            "label": ent.label_
        }
        for ent in doc.ents
        if ent.label_ in ENTITY_TYPES
    ]

In [ ]:
test_df["ner_entities"] = (
    test_df["cleaned_body"]
    .fillna("")
    .apply(extract_entities)
)

print(
    "Baseline entities detected:",
    sum(len(x) for x in test_df["ner_entities"])
)

## 4. Enron-Specific PERSON Extraction

Enron emails frequently contain directory-style names such as:

`Andrew S Fastow/HOU/ECT@ECT`

A generic NER model may incorrectly treat the entire directory expression as an entity.  
The following rule extracts the person's name and records its character span so the complete directory expression can be masked before spaCy processes the email.

In [ ]:
import re

def extract_enron_directory_v4(text):
    """
    Robust Enron directory PERSON extraction.

    Handles examples such as:
        Andrew S Fastow/HOU/ECT@ECT
        Steven J Kean/NA/Enron@Enron
        David W Delainey/HOU/ECT@ECT
        Mark E Haedicke/HOU/ECT@ECT
        William S Bradford/HOU/ECT@ECT
        Karen S Owens@ees@EES

    Also handles names broken across line breaks.
    """

    if not isinstance(text, str):
        return []

    # Allow spaces/newlines between name components
    ws = r"[\s]+"

    # A name consists of 2–5 words.
    # Each word starts with a capital letter.
    name = (
        r"([A-Z][A-Za-z.'-]*"
        rf"(?:{ws}[A-Z][A-Za-z.'-]*)"
        r"{1,4})"
    )

    # Directory suffix:
    # /HOU
    # /HOU/ECT
    # /Corp/Enron@Enron
    # @ees@EES
    suffix = r"(?:/[A-Za-z0-9_&.-]+)*(?:@[A-Za-z0-9_&.-]+)+"

    # Also allow directory paths without @
    suffix_without_email = r"(?:/[A-Za-z0-9_&.-]+)+"

    patterns = [
        # Name/HOU/ECT@ECT
        name + r"[\s]*" + suffix,

        # Name/HOU
        name + r"[\s]*" + suffix_without_email
    ]

    results = []

    for pattern in patterns:
        for match in re.finditer(pattern, text):
            person_name = re.sub(r"\s+", " ", match.group(1)).strip()

            if len(person_name.split()) >= 2:
                results.append({
                    "text": person_name,
                    "label": "PERSON",
                    "start": match.start(),
                    "end": match.end()
                })

    # Remove overlapping duplicates.
    # Keep the longest match when they start at the same position.
    results.sort(
        key=lambda x: (
            x["start"],
            -(x["end"] - x["start"])
        )
    )

    selected = []

    for entity in results:
        overlaps = False

        for existing in selected:
            if (
                entity["start"] < existing["end"]
                and existing["start"] < entity["end"]
            ):
                overlaps = True
                break

        if not overlaps:
            selected.append(entity)

    return selected

## 5. Final Domain-Adapted NER Pipeline

In [ ]:
def final_ner_pipeline(text):
    """
    Domain-adapted NER pipeline:

    1. Extract Enron directory-style PERSON entities
    2. Mask those directory expressions
    3. Run spaCy NER on the remaining text
    4. Combine both results
    5. Remove duplicates
    """

    if not isinstance(text, str):
        return []

    # -----------------------------------
    # 1. Extract Enron-specific entities
    # -----------------------------------
    enron_entities = extract_enron_directory_v4(text)

    # -----------------------------------
    # 2. Mask Enron directory expressions
    # -----------------------------------
    masked_text = text

    # Process from right to left so character
    # positions remain valid
    for entity in sorted(
        enron_entities,
        key=lambda x: x["start"],
        reverse=True
    ):
        start = entity["start"]
        end = entity["end"]

        masked_text = (
            masked_text[:start]
            + " " * (end - start)
            + masked_text[end:]
        )

    # -----------------------------------
    # 3. Run spaCy on masked text
    # -----------------------------------
    doc = nlp(masked_text)

    spacy_entities = []

    for ent in doc.ents:
        if ent.label_ in ENTITY_TYPES:
            spacy_entities.append({
                "text": clean_entity_text(ent.text),
                "label": ent.label_
            })

    # -----------------------------------
    # 4. Combine Enron + spaCy entities
    # -----------------------------------
    combined = spacy_entities + [
        {
            "text": entity["text"],
            "label": "PERSON"
        }
        for entity in enron_entities
    ]

    # -----------------------------------
    # 5. Remove duplicates
    # -----------------------------------
    final_entities = []
    seen = set()

    for entity in combined:
        text_clean = clean_entity_text(entity["text"])

        key = (
            text_clean.lower(),
            entity["label"]
        )

        if text_clean and key not in seen:
            seen.add(key)

            final_entities.append({
                "text": text_clean,
                "label": entity["label"]
            })

    return final_entities

In [ ]:
test_df["domain_adapted_ner"] = (
    test_df["cleaned_body"]
    .fillna("")
    .apply(final_ner_pipeline)
)

test_df["domain_adapted_ner_output"] = (
    test_df["domain_adapted_ner"]
    .apply(format_entities)
)

print("Domain-adapted entities detected:",
      sum(len(x) for x in test_df["domain_adapted_ner"]))

In [ ]:
# Example final output
print(test_df.iloc[50]["domain_adapted_ner_output"])

## 6. Baseline vs. Domain-Adapted NER

In [109]:
def count_entities(entity_list):
    return len(entity_list)

def count_label(entity_list, label):
    return sum(
        1 for entity in entity_list
        if entity["label"] == label
    )

comparison = {
    "Baseline Total Entities": sum(
        len(x) for x in test_df["ner_entities"]
    ),

    "Domain-Adapted Total Entities": sum(
        len(x) for x in test_df["domain_adapted_ner"]
    ),

    "Baseline PERSON": sum(
        count_label(x, "PERSON")
        for x in test_df["ner_entities"]
    ),

    "Domain-Adapted PERSON": sum(
        count_label(x, "PERSON")
        for x in test_df["domain_adapted_ner"]
    ),

    "Emails in Test Set": len(test_df),

    "Baseline Emails with Entities": sum(
        1 for x in test_df["ner_entities"]
        if len(x) > 0
    ),

    "Domain-Adapted Emails with Entities": sum(
        1 for x in test_df["domain_adapted_ner"]
        if len(x) > 0
    )
}

for key, value in comparison.items():
    print(f"{key}: {value}")

Baseline Total Entities: 3023
Domain-Adapted Total Entities: 2203
Baseline PERSON: 1031
Domain-Adapted PERSON: 852
Emails in Test Set: 90
Baseline Emails with Entities: 87
Domain-Adapted Emails with Entities: 87


## 7. Ground Truth

In [122]:
ground_truth = [(0, 'Peter Keohane', 'PERSON'), (0, 'Soma Ghosh', 'PERSON'), (0, 'Clement Abrams', 'PERSON'), (0, 'Tana Jones', 'PERSON'), (0, 'Sara Shackleton', 'PERSON'), (0, 'William S Bradford', 'PERSON'), (0, 'Derek Davies', 'PERSON'), (0, 'Brian Kerrigan', 'PERSON'), (0, 'Greg Johnston', 'PERSON'), (0, 'Sharon Crawford', 'PERSON'), (0, 'RBC', 'ORG'), (0, 'ECT', 'ORG'), (0, 'Bank', 'ORG'), (0, 'Blakes', 'ORG'), (0, 'Enron Corp.', 'ORG'), (0, 'Macleod Dixon', 'ORG'), (0, 'Swapco', 'ORG'), (0, 'ECC', 'ORG'), (0, 'ECPC', 'ORG'), (0, 'Alberta', 'GPE'), (0, 'Friday', 'DATE'), (0, 'tomorrow', 'DATE'), (0, 'Tuesday', 'DATE'), (0, 'Wednesday', 'DATE'), (0, 'Thursday', 'DATE'), (0, 'mid-afternoon', 'TIME'), (1, 'Tom', 'PERSON'), (1, 'Sarah Novosel', 'PERSON'), (1, 'Kevin M Presto', 'PERSON'), (1, 'Mark Dana Davis', 'PERSON'), (1, 'Jeff Ader', 'PERSON'), (1, 'Edward D Baughman', 'PERSON'), (1, 'Joe Gordon', 'PERSON'), (1, 'Janelle Scheuer', 'PERSON'), (1, 'Mark Bernstein', 'PERSON'), (1, 'John Llodra', 'PERSON'), (1, 'George Wood', 'PERSON'), (1, 'Paul J Broderick', 'PERSON'), (1, 'Jason Thompkins', 'PERSON'), (1, 'Mason Hamlin', 'PERSON'), (1, 'Robert Stalford', 'PERSON'), (1, 'Tom May', 'PERSON'), (1, 'Gautam Gupta', 'PERSON'), (1, 'Narsimha Misra', 'PERSON'), (1, 'Steve Montovano', 'PERSON'), (1, 'Garrett Tripp', 'PERSON'), (1, 'Berney C Aucoin', 'PERSON'), (1, 'Rob Wheeler', 'PERSON'), (1, 'Jim Meyn', 'PERSON'), (1, 'Aleck Dadson', 'PERSON'), (1, 'Daniel Allegretti', 'PERSON'), (1, 'Pearce W Hammond', 'PERSON'), (1, 'Joe Hartsoe', 'PERSON'), (1, 'Donna Fulton', 'PERSON'), (1, 'Howard Fromer', 'PERSON'), (1, 'Kathleen Sullivan', 'PERSON'), (1, 'Tom Hoatson', 'PERSON'), (1, 'Thane Twiggs', 'PERSON'), (1, 'Sarah Novosel', 'PERSON'), (1, 'Christi L Nicolay', 'PERSON'), (1, 'James D Steffes', 'PERSON'), (1, 'Linda Robertson', 'PERSON'), (1, 'Richard Shapiro', 'PERSON'), (1, 'Steven J Kean', 'PERSON'), (1, 'Charles Decker', 'PERSON'), (1, 'FERC', 'ORG'), (1, 'NEPOOL', 'ORG'), (1, 'PJM', 'ORG'), (1, 'New York', 'GPE'), (1, 'NY', 'GPE'), (1, 'April 25', 'DATE'), (1, 'July 1, 2001', 'DATE'), (1, 'April 13', 'DATE'), (1, 'hourly', 'TIME'), (2, 'John', 'PERSON'), (2, 'One Great Night', 'EVENT'), (2, 'Nov 8', 'DATE'), (2, 'Last year', 'DATE'), (3, 'Mark', 'PERSON'), (3, 'Todd', 'PERSON'), (3, 'Mark Guzman', 'PERSON'), (3, 'Friday', 'DATE'), (3, 'June 22', 'DATE'), (3, 'STCali', 'ORG'), (3, 'STWHourly', 'ORG'), (3, 'Cali', 'GPE'), (3, 'Monday', 'DATE'), (3, 'Risk', 'ORG'), (4, 'Wednesday', 'DATE'), (4, '11:30 a.m.', 'TIME'), (4, 'Friday', 'DATE'), (4, 'July 28,2000', 'DATE'), (4, 'Pat', 'PERSON'), (4, 'Steve', 'PERSON'), (5, 'Sally Beck', 'PERSON'), (5, 'Brent Price', 'PERSON'), (5, 'Rick Causey', 'PERSON'), (5, 'Kimberly Rizzi', 'PERSON'), (5, 'Jennifer Jordan', 'PERSON'), (5, 'Donald Miller', 'PERSON'), (5, 'Sheila Walton', 'PERSON'), (5, 'Enron North America Corp.', 'ORG'), (5, 'Enron', 'ORG'), (5, 'December 8th', 'DATE'), (5, 'December 12th', 'DATE'), (5, 'November 29th', 'DATE'), (5, 'Wednesday', 'DATE'), (5, '11/27/2000', 'DATE'), (6, 'Dana Jones', 'PERSON'), (6, 'Drew Fossum', 'PERSON'), (6, 'Stan Horton', 'PERSON'), (6, 'Michael Moran', 'PERSON'), (6, 'Mike', 'PERSON'), (6, 'Enron', 'ORG'), (6, 'Enron Transportation Services', 'ORG'), (6, 'Northern Natural Gas', 'ORG'), (6, 'Transwestern', 'ORG'), (6, 'ETS', 'ORG'), (6, 'Houston', 'GPE'), (6, 'April 1, 2001', 'DATE'), (6, 'March 07, 2001', 'DATE'), (7, 'Jeffrey C Gossett', 'PERSON'), (7, 'Larry Joe Hunter', 'PERSON'), (7, 'Julie Brewer', 'PERSON'), (7, 'Joe Hunter', 'PERSON'), (7, 'Tana Jones', 'PERSON'), (7, 'Kim S Theriot', 'PERSON'), (7, 'Mark Greenberg', 'PERSON'), (7, 'Mark Taylor', 'PERSON'), (7, 'Jeffrey T Hodge', 'PERSON'), (7, 'Stacy E Dickson', 'PERSON'), (7, 'Marcus Nettelton', 'PERSON'), (7, 'Elizabeth Sager', 'PERSON'), (7, 'Alan Aronowitz', 'PERSON'), (7, 'EnronOnline', 'ORG'), (7, 'TAGG', 'ORG'), (7, 'Legal Department', 'ORG'), (7, 'Enron', 'ORG'), (7, 'April 30, 2001', 'DATE'), (7, 'April 27, 2001', 'DATE'), (8, 'Mahesh Lakhani', 'PERSON'), (8, 'Sara Shackleton', 'PERSON'), (8, 'Cindy Buckley', 'PERSON'), (8, 'Paul Simons', 'PERSON'), (8, 'John Greene', 'PERSON'), (8, 'Janine Juggins', 'PERSON'), (8, 'Enron Investment Services Ltd', 'ORG'), (8, 'ENA', 'ORG'), (8, 'ECT Investments Inc.', 'ORG'), (8, 'UK', 'GPE'), (8, '10/06/2000', 'DATE'), (8, '04/10/2000', 'DATE'), (8, '10/04/2000', 'DATE'), (8, '3/10/00', 'DATE'), (8, 'U.K.', 'GPE')]

ground_truth_df = pd.DataFrame(
    ground_truth,
    columns=["email_id", "entity", "label"]
)

print("Ground-truth entities:", len(ground_truth_df))
display(ground_truth_df.head(20))

Ground-truth entities: 155


,email_id,entity,label
0,0,Peter Keohane,PERSON
1,0,Soma Ghosh,PERSON
2,0,Clement Abrams,PERSON
3,0,Tana Jones,PERSON
4,0,Sara Shackleton,PERSON
5,0,William S Bradford,PERSON
6,0,Derek Davies,PERSON
7,0,Brian Kerrigan,PERSON
8,0,Greg Johnston,PERSON
9,0,Sharon Crawford,PERSON


## 8. Precision, Recall & F1 Evaluation

In [123]:
import re
from collections import Counter

def normalize_entity(text):
    """
    Normalize entity text for evaluation.
    """
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip().lower()
    return text


def get_prediction_set(entities):
    """
    Convert model entity list into normalized (text, label) pairs.
    """
    result = set()

    for entity in entities:
        text = normalize_entity(entity["text"])
        label = entity["label"]

        if text:
            result.add((text, label))

    return result


def get_ground_truth_set(df):
    """
    Convert ground truth dataframe into normalized
    (email_id, entity, label) tuples.
    """
    result = set()

    for _, row in df.iterrows():
        result.add((
            int(row["email_id"]),
            normalize_entity(row["entity"]),
            row["label"]
        ))

    return result


def evaluate_ner(test_df, ground_truth_df, prediction_column):
    """
    Evaluate NER using exact normalized entity text + label.
    """

    # Ground truth
    true_set = get_ground_truth_set(ground_truth_df)

    # Predictions
    pred_set = set()

    for email_id, entities in enumerate(test_df[prediction_column]):
        for entity in entities:

            text = normalize_entity(entity["text"])
            label = entity["label"]

            if text:
                pred_set.add(
                    (email_id, text, label)
                )

    # Calculate TP / FP / FN
    true_positive = true_set & pred_set
    false_positive = pred_set - true_set
    false_negative = true_set - pred_set

    tp = len(true_positive)
    fp = len(false_positive)
    fn = len(false_negative)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "true_set": true_set,
        "pred_set": pred_set,
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative
    }


# Evaluate baseline
baseline_results = evaluate_ner(
    test_df,
    ground_truth_df,
    "ner_entities"
)

# Evaluate domain-adapted NER
adapted_results = evaluate_ner(
    test_df,
    ground_truth_df,
    "domain_adapted_ner"
)


print("=" * 70)
print("BASELINE spaCy NER")
print("=" * 70)

print("True Positives :", baseline_results["TP"])
print("False Positives:", baseline_results["FP"])
print("False Negatives:", baseline_results["FN"])
print("Precision       :", round(baseline_results["Precision"], 4))
print("Recall          :", round(baseline_results["Recall"], 4))
print("F1 Score        :", round(baseline_results["F1"], 4))


print("\n" + "=" * 70)
print("DOMAIN-ADAPTED NER")
print("=" * 70)

print("True Positives :", adapted_results["TP"])
print("False Positives:", adapted_results["FP"])
print("False Negatives:", adapted_results["FN"])
print("Precision       :", round(adapted_results["Precision"], 4))
print("Recall          :", round(adapted_results["Recall"], 4))
print("F1 Score        :", round(adapted_results["F1"], 4))

BASELINE spaCy NER
True Positives : 57
False Positives: 2177
False Negatives: 97
Precision       : 0.0255
Recall          : 0.3701
F1 Score        : 0.0477

DOMAIN-ADAPTED NER
True Positives : 115
False Positives: 2088
False Negatives: 39
Precision       : 0.0522
Recall          : 0.7468
F1 Score        : 0.0976


## 9. Final Result

The manually evaluated test set contains **155 ground-truth entities**.

The domain-adapted pipeline improves recall and F1 compared with the baseline spaCy NER:

| Model | Precision | Recall | F1 |
|---|---:|---:|---:|
| Baseline spaCy NER | 0.0255 | 0.3701 | 0.0477 |
| Domain-adapted NER | 0.0522 | 0.7468 | 0.0976 |

The main improvement comes from handling Enron-specific directory-style PERSON names that the generic spaCy model struggles to recognize correctly.